# DroneRL — Research Analysis Notebook

This notebook reproduces the experimental evidence behind the claims in
[`docs/assignment-2/EXPERIMENTS.md`](../docs/assignment-2/EXPERIMENTS.md)
and the cost figures in
[`docs/assignment-2/COST_ANALYSIS.md`](../docs/assignment-2/COST_ANALYSIS.md).

It satisfies the course-wide submission guidelines (§9.2 *Results
Analysis Notebook*) by running the analysis end-to-end inside a
notebook, displaying every chart inline, and surfacing every measured
number from a committed script.

**Running:** `uv run jupyter notebook notebooks/research_analysis.ipynb`
or open it in VS Code with the Python kernel.

---

## 1. Setup — repo root on the Python path

The analysis modules are not yet pip-installed, so put the repo root
on `sys.path` so `import analysis` works from the `notebooks/`
directory.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
    sys.path.insert(0, str(ROOT))

print(f'repo root: {ROOT}')
print(f'analysis on path: {(ROOT / "analysis").exists()}')

## 2. Multi-seed robustness

5 seeds × 1500 episodes × 3 algorithms on the medium-difficulty board.
Output is a chart with mean ± 95 % CI bands per algorithm.

**Hypothesis (H1):** Bellman fails under noise — its last-200 mean
should be clearly worse than Q-Learning and Double-Q at noise = 0.5.

**What we actually find** (see EXPERIMENTS.md §H1): Bellman ties
Q-Learning at this difficulty; the surprise is that **Double-Q is the
seed-sensitive one** at 1500 episodes — 2 of 5 seeds collapse.

In [ ]:
from analysis.multi_seed_robustness import main as run_multi_seed
run_multi_seed()

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'data' / 'analysis' / 'multi_seed_robustness.png')))

## 3. Alpha-decay sensitivity sweep

6 decay values × 3 seeds × {Q-Learning, Double-Q}, with Bellman as a
horizontal reference line. Tests whether the conclusions about
Q-Learning vs. Bellman are robust to the precise decay value.

**Hypothesis (H2):** Q-Learning's advantage should hold across a
range of decay rates.

**What we actually find** (see EXPERIMENTS.md §H2): Q-Learning is
flat across the entire decay grid (76 → 78). At this difficulty and
training budget, the decay parameter barely matters.

In [ ]:
from analysis.alpha_decay_sweep import main as run_decay_sweep
run_decay_sweep()

In [ ]:
display(Image(filename=str(ROOT / 'data' / 'analysis' / 'alpha_decay_sweep.png')))

## 4. Cost profile

Wall-time, peak heap, and Q-table bytes for each algorithm.
Underpins the runtime cost story in COST_ANALYSIS.md §1–3.

In [ ]:
from analysis.cost_profile import main as run_cost_profile
run_cost_profile()

In [ ]:
import json
with open(ROOT / 'data' / 'analysis' / 'cost_profile.json') as f:
    cost = json.load(f)
for p in cost['profiles']:
    print(f"  {p['algorithm']:12s}  {p['elapsed_s']:6.3f} s  "
          f"{p['episodes_per_s']:6.1f} ep/s  "
          f"{p['us_per_episode']:6.1f} us/ep  "
          f"Q-table {p['q_table_bytes']} bytes")

## 5. Conclusions

The qualitative picture in the README's *Conclusions* section holds,
but with two caveats this notebook surfaces directly:

1. **Double-Q is seed-sensitive at short training budgets.** Its
   bias-correction benefit only emerges with enough samples to
   dominate the higher initial variance from running two interleaved
   Q-tables. At Scenario 2's 6 000-episode budget the textbook
   ordering reappears.
2. **Q-Learning's α-decay barely moves the needle** at this difficulty.
   The decay separates Q-Learning from Bellman *in theory* (Σα² < ∞);
   *in practice* the same plateau is reached even at decay = 1.0.

Full hypothesis-by-hypothesis write-up:
[`docs/assignment-2/EXPERIMENTS.md`](../docs/assignment-2/EXPERIMENTS.md).

Cost analysis (runtime + AI-development cost):
[`docs/assignment-2/COST_ANALYSIS.md`](../docs/assignment-2/COST_ANALYSIS.md).